# Notebook 22a — Audit by Group: Accuracy for Whom?
**AI Models and Agents for Business · Week 11 · Session 22**

Gender Shades found that commercial face-analysis systems that looked accurate overall made far more mistakes for some groups than for others. Today you run the same kind of audit on a fictional vendor's system, in about twelve minutes.

**The setting:** a bank is considering a vendor's **automatic ID check** for new customers who open accounts online. The system compares the selfie with the ID photo and **accepts** or **rejects** the applicant. The vendor's slide says: *"92% accurate."* You have the vendor's own test results: 1,000 applicants, what the system decided, and what was true.

**No API key needed.** The data are synthetic and the vendor is fictional.

### What you will do
1. **Compute** the error rate overall and for each group.
2. **Compare:** which group is worst, and what kind of mistake is it?
3. **Write** one sentence a manager could act on.
4. Check **who is in the test set**, and look at the groups **within** groups.

In [ ]:
# Cell 1 — The vendor's test results. Run this first.
import pandas as pd
import numpy as np

# (group, age band, genuine customers, impostors, genuine customers wrongly rejected, impostors wrongly accepted)
CELLS = [("A", "under 40", 485, 65, 25, 7), ("A", "40 and over", 265, 35, 15, 4),
         ("B", "under 40",  78, 12,  9, 1), ("B", "40 and over",  52,  8, 17, 1)]
rows = []
for grp, age, genuine, impostor, false_rej, false_acc in CELLS:
    rows += [(grp, age, 1, 0)] * false_rej + [(grp, age, 1, 1)] * (genuine - false_rej)
    rows += [(grp, age, 0, 1)] * false_acc + [(grp, age, 0, 0)] * (impostor - false_acc)
results = pd.DataFrame(rows, columns=["group", "age_band", "genuine", "accepted"]).sample(frac=1, random_state=22)
results.insert(0, "applicant", range(1, len(results) + 1))
results["correct"] = results.genuine == results.accepted

print(f"Applicants in the vendor's test: {len(results):,}")
print(f"Overall accuracy: {results.correct.mean():.1%}   <- the number on the vendor's slide")
results.head(8)

---
## Part A — Compute: the error rate for each group, and overall

Two kinds of mistake:
- **False rejection:** a genuine customer is rejected. They cannot open the account, or wait for a manual check.
- **False acceptance:** an impostor is accepted. The bank takes on fraud.

In [ ]:
# Cell 2 — Error rates, overall and by group
def audit(df, by=None):
    groups = [("Everyone", df)] if by is None else [("Everyone", df)] + list(df.groupby(by))
    out = []
    for name, sub in groups:
        genuine, impostor = sub[sub.genuine == 1], sub[sub.genuine == 0]
        out.append({
            "Who": name if isinstance(name, str) else " / ".join(name),
            "Applicants": len(sub),
            "Error rate": 1 - sub.correct.mean(),
            "False rejection rate": (genuine.accepted == 0).mean(),
            "False acceptance rate": (impostor.accepted == 1).mean(),
        })
    return pd.DataFrame(out).set_index("Who")

pct = lambda t: t.style.format({c: "{:.1%}" for c in t.columns if "rate" in c})
display(pct(audit(results, by="group")))

---
## Part B — Compare

💬 **Discuss (pairs):**
- Which group is worst? By how much, compared with the overall number?
- What **kind** of error is it, mostly? Who bears that cost: the bank, or the customer?
- Why did the overall error rate look fine? (Look at the **Applicants** column.)

---
## Part C — ✏️ YOUR TURN: write one sentence a manager could act on

A good finding names the group, the size of the gap and the kind of mistake, and says what to do.

- **Good:** *"Overall error is 8%, but 19% for group B, mostly genuine customers wrongly rejected. Do not deploy until the vendor fixes that."*
- **Weak:** *"The system is 92% accurate."*

> *Our finding:* ...

---
## Part D — Who is in the test set?

The vendor tested on 1,000 applicants, but only 15% were in group B. Suppose **40%** of *your* bank's online applicants are in group B. The overall number the vendor showed you is not the number your customers will get.

In [ ]:
# Cell 3 — ✏️ YOUR TURN: re-weight to your own customers
YOUR_SHARE_B = 0.40          # share of YOUR applicants who are in group B (try 0.15, 0.40, 0.60)

by_group = audit(results, by="group")
vendor_share_b = (results.group == "B").mean()
expected = (1 - YOUR_SHARE_B) * by_group.loc["A", "Error rate"] + YOUR_SHARE_B * by_group.loc["B", "Error rate"]

print(f"Share of group B in the vendor's test: {vendor_share_b:.0%}")
print(f"Overall error the vendor reported:     {1 - results.correct.mean():.1%}")
print(f"Overall error with {YOUR_SHARE_B:.0%} group B:        {expected:.1%}")

💬 **Discuss:** Gender Shades made the same point: the usual test sets were mostly lighter-skinned men, so overall accuracy looked fine. What would you ask the vendor about its test set **before** you look at its accuracy?

---
## Part E — Groups within groups

Gender Shades found the worst results for one **combination** of groups (darker-skinned women), not for either group alone. Split by group **and** age band.

In [ ]:
# Cell 4 — Group and age band together
display(pct(audit(results, by=["group", "age_band"])))

💬 **Discuss:** Which combination is worst? Would you have found it if you had audited group and age separately? How many applicants are in that cell, and how sure can you be about its rate?

## Carry this into Project 3
When you audit another team's assistant, report results **by type of question or user**, not only overall, and say how many cases sit behind each number.

---
## ✅ Notebook 22a complete — Reflection (submit with your notebook)

These prep you for **Quiz 6** (Session 25).

1. Your one-sentence finding from Part C.
2. Why did the vendor's overall accuracy hide the problem? Give two reasons (one is in Part D).
3. Which kind of mistake (false rejection or false acceptance) matters more for this bank, and for its customers? Are those the same answer?
4. What is the smallest cell in Part E, and why should that make you careful about its error rate?

*Next: Notebook 22b, overfitting.*